# Async LLM usage

In [1]:
import json
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
fs = sorted(Path("../data/json3").glob("*.json"))
print(f"{len(fs)} threads")

10266 threads


In [3]:
from IPython.display import display
# load output from
df3 = pd.read_parquet("../outputs/df_links4.parquet")
df3

,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url
title,,,,,,,,
Worth the Candle,1460,110,110,"[{'author': 'Noumero', 'author_flair_text': 'S...",[https://reddit.com/r/rational/comments/7dz6kj...,2017-07-28 13:17:36,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...
"Alexander Wales - The Metropolitan Man, Shadows of the Limelight",1402,22,22,"[{'author': 'alexanderwales', 'author_flair_te...",[https://reddit.com/r/rational/comments/al7z2v...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales]
Mother of Learning,1063,101,101,"[{'author': 'TimeLoopedPowerGamer', 'author_fl...",[https://reddit.com/r/rational/comments/cmc4a0...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...
A Practical Guide to Evil,1006,92,92,"[{'author': 'Escapement', 'author_flair_text':...",[https://reddit.com/r/rational/comments/byyy3d...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ..."
Worm,916,83,83,"[{'author': 'ArgentStonecutter', 'author_flair...",[https://reddit.com/r/rational/comments/16rsx5...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa..."
...,...,...,...,...,...,...,...,...
Much less than or Not at all.,-7,1,1,"[{'author': 'Blusqere', 'author_flair_text': N...",[https://reddit.com/r/rational/comments/kaat79...,2020-12-10 16:35:49,2020-12-10 16:35:49,[https://www.merriam-webster.com/dictionary/no...
"Steelheart (The Reckoners): 9780385743570: Sanderson, Brandon: Books",-10,1,1,"[{'author': 'ben_oni', 'author_flair_text': No...",[https://reddit.com/r/rational/comments/7cneg5...,2017-11-13 20:58:01,2017-11-13 20:58:01,[https://www.amazon.com/Steelheart-Reckoners-B...
Orthogonality thesis,-10,2,2,"[{'author': 'BadGoyWithAGun', 'author_flair_te...",[https://reddit.com/r/rational/comments/3vc0si...,2015-12-04 18:17:07,2016-07-11 16:27:30,[https://wiki.lesswrong.com/wiki/Orthogonality...


## Extra get a llm summary of each link [WIP]

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [4]:
import dotenv
import os

dotenv.load_dotenv()
from openai import OpenAI
from openai import AsyncOpenAI
client = AsyncOpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ['OPENROUTER_API_KEY'],
)

import tiktoken
# https://openrouter.ai/models?fmt=cards&order=newest&supported_parameters=structured_outputs
MODEL_NAME = "gpt-4o-mini" # ?b
cost = 0.150 / 1e6
# MODEL_NAME = "google/gemini-flash-1.5" # json fail... should work though
MODEL_NAME = "google/gemini-flash-1.5" 
# MODEL_NAME = "qwen/qwen-2.5-72b-instruct"
# MODEL_NAME = "cohere/command-r" # json fail
# MODEL_NAME = "meta-llama/llama-3.3-70b-instruct" #json fail
enc = tiktoken.encoding_for_model('gpt-4')

In [5]:
# load all md posts 
md_posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in tqdm(fs):
    s = f.open().read()
    md_posts.append(s)


# order by date
def md2date(s: str) -> str:
    return s.split('* Created: ')[1].split('\n')[0]

md_posts = sorted(md_posts, key=md2date)
md2date(md_posts[0]), md2date(md_posts[-1])

  0%|          | 0/10266 [00:00<?, ?it/s]

('2009-11-25T02:34:03', '2024-12-30T15:00:12')

In [6]:

def get_post_context(urls: List[str], char_budget=100000, min_size=1000) -> str:
    assert len(urls) > 0

    matches = []
    for ii, post in enumerate(md_posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = char_budget / len(matches)
    budget_pp = max(budget_pp, min_size)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk

    # if too large get first N//2 and last N//2
    if len(s) > char_budget:
        s = s[:char_budget // 2] + "..." + s[-char_budget // 2:]
    return s


In [7]:


from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str = Field(description="Title of the fiction")
    description: str = Field(description="A few paragraphs of very concise, informative, dense, description of the fiction")
    tags: List[str] = Field(
        description="""Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: str = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote excerpts from every single readers' comments about the fiction"
    )
    reviews_summary: str = Field(
        description="Structured, dry, and concise summary of reviews"
    )
    reccomendations: str = Field(
        description="Why readers recommend the fiction"
    )
    disrecommendations: str = Field(
        description="Why readers disrecommend the fiction"
    )
    why: str = Field(
        description="Why/when might readers of r/rational like the fiction"
    )
    if_you_liked_x_you_will_like_this: List[str] = Field(
        description="Fans of X will also like the fiction. List all examples that readers mention wrt to the fiction."
    )

    rating_quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment of readers towards the fiction (out of 10), it's important to be consistent and use the same scale for all fictions"
    )
    rating_rationality: float = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: float
    rating_plot: float
    rating_character: float
    rating_worldbuilding: float

    # TODO I want to know, how long is it, is it complete, is it a fiction, article, comic, etc.

In [8]:
from openai.lib._pydantic import to_strict_json_schema

schema = to_strict_json_schema(FictionInfo)
schema = json.dumps(schema)
# print(schema)

In [9]:

async def get_llm_summary(title: str, urls: str, context: str):
    chat_completion = await client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": f"You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community recommendations from r/rational into a dry, informative, concise, and structured format for your own personal notes. Because it's private you can be consise, frank, and opinionated. Ignoring any authors promotion. You  answer in JSON. Here's the json schema you must adhere to:\n<schema>\n{schema}\n",
            },
            {
                "role": "user",
                "content": f"""Using the given structure, summarize the parts of the discussion which talk about the fiction: {title} (urls: {urls}).

### Discussion:

{context}

### Instructions

You are Gwern Branwern, using the given structure, summarize the above discussion of the fiction {title} (urls: {urls}).""",
            },
        ],
        model=MODEL_NAME,
        response_format=FictionInfo,
    )

    tokens = chat_completion.usage.prompt_tokens
    return tokens, chat_completion.choices[0].message.parsed.__dict__

In [10]:
model_name_ds = re.sub(r'[^\w\s]', '', MODEL_NAME)

output_dir = Path(f'../outputs/llm2/{model_name_ds}')
output_dir.mkdir(exist_ok=True, parents=True)
output_dir

PosixPath('../outputs/llm2/googlegeminiflash15')

In [11]:
# import shutil
# shutil.rmtree(output_dir, ignore_errors=True)

from pydantic import ValidationError

In [12]:
import asyncio
import json
import os


In [13]:
import asyncio

async def process_item(title, urls, f):
    if f.exists():
        return json.load(f.open())
    context = get_post_context(urls, char_budget=120000)
    try:
        f_tokens, llm_data = await get_llm_summary(title, urls, context)
    except ValidationError as e:
        print(e, f)
        return None
    except ValueError as e:
        print(e, f)
        return None
    

    llm_data['context'] = context
    llm_data['title2'] = title
    llm_data['model'] = MODEL_NAME
    
    # print(
    #     f"Input Tokens: {f_tokens}. Input Cost: {cost * f_tokens:.6f} USD, for title=`{title}`"
    # )
    with f.open("w") as fo:
        json.dump(llm_data, fo)


    return llm_data

In [14]:

# def main(df3, batch_size=1000):
#     all_llm_info = []
#     total_tasks = len(df3)
#     cached = []

#     for start in range(0, total_tasks, batch_size):
#         end = min(start + batch_size, total_tasks)
#         batch = df3.iloc[start:end]
        
#         tasks = []
#         for i, (title, row) in enumerate(batch.iterrows(), start=start):
#             urls = row.url
#             fs_title = re.sub(r'[^\w\s]', '', title)[:200]
#             f = output_dir / f"{i}_{fs_title}.json"
#             if f.exists():
#                 d = json.load(f.open())
#                 cached.append(d)
#             # tasks.append(process_item(title, urls, f))

#         # batch_llm_info = await asyncio.gather(*tasks)
#         # all_llm_info.extend(batch_llm_info)
        
#         print(f"Processed batch {start//batch_size + 1} of {(total_tasks - 1)//batch_size + 1}")

#     print("All batches processed")
#     return cached #+ all_llm_info
# d = main(df3)
# len(d), len(df3)

In [26]:
# async def main(df3):
#     tasks = []

#     for i in range(len(df3)):
#         title = df3.index[i]
#         urls = df3.iloc[i].url

#         fs_title = re.sub(r'[^\w\s]', '', title)[:200]
        
#         f = output_dir / f"{i}_{fs_title}.json"
#         tasks.append(process_item(title, urls, f))

#     llm_info = await asyncio.gather(*tasks)
#     print(llm_info)
#     return llm_info

async def main(df3, batch_size=1000):
    all_llm_info = []
    total_tasks = len(df3)
    cached = []

    for start in range(0, total_tasks, batch_size):
        end = min(start + batch_size, total_tasks)
        batch = df3.iloc[start:end]
        
        tasks = []
        for i, (title, row) in enumerate(batch.iterrows(), start=start):
            urls = row.url
            fs_title = re.sub(r'[^\w\s]', '', title)[:200]
            f = output_dir / f"{i}_{fs_title}.json"
            if f.exists():
                d = json.load(f.open())
                cached.append(d)
            tasks.append(process_item(title, urls, f))

        batch_llm_info = await asyncio.gather(*tasks)
        failed = [1 if d is None else 0 for d in batch_llm_info]
        all_llm_info.extend(batch_llm_info)
        
        print(f"Processed batch {start//batch_size + 1} of {(total_tasks - 1)//batch_size + 1}")
        print(f"\t{len(cached)} cached, {len(tasks)} tasks, failed: {sum(failed)}")

    print("All batches processed")
    return cached + all_llm_info

# Run the async main function
df4 = df3#.sample(200, random_state=42)
loop = asyncio.get_event_loop()
task = loop.create_task(main(df4))
value = task.result()
value

InvalidStateError: Result is not set.

Processed batch 1 of 16
	996 cached, 1000 tasks, failed: 0
Processed batch 2 of 16
	1995 cached, 1000 tasks, failed: 0
Processed batch 3 of 16
	2994 cached, 1000 tasks, failed: 0
1 validation error for FictionInfo
  Invalid JSON: EOF while parsing a string at line 1 column 1086 [type=json_invalid, input_value='{"description": "A twitt...d chuckle out of me, at', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid ../outputs/llm2/googlegeminiflash15/3289_httpstwittercomhillelogram.json
1 validation error for FictionInfo
  Invalid JSON: EOF while parsing a string at line 1 column 2246 [type=json_invalid, input_value='{"description": "The con...rry, I meant meaningful', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid ../outputs/llm2/googlegeminiflash15/3605_Calendar of saints.json
Processed batch 4 of 16
	3987 cached, 1000 tasks, failed: 2
1 validation error for FictionInfo
  Invalid JSON: EOF 

In [110]:
llm_info = task.result()
llm_info = [x for x in llm_info if x is not None]
df_llm = pd.DataFrame(llm_info)
df_llm


,title,description,tags,reviews_quotes,reviews_summary,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,rating_quality,rating_rationality,rating_writing,rating_plot,rating_character,rating_worldbuilding,context,title2,model
0,Worth the Candle,"Worth the Candle is a long, original, complete...","[web serial, LitRPG, isekai, portal fantasy, o...","[This is a self-insert litRPG portal fantasy, ...","Overall, Worth the Candle received overwhelmin...",Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,Readers of r/rational would likely enjoy Worth...,"[Mother of Learning, HPMOR, Worm, Erfworld, Lu...",9.0,7.0,8.5,8.5,8.0,9.0,\n\n----- Thread 10262 -----\n\n## [RT][WIP] W...,Worth the Candle,google/gemini-flash-1.5
1,"Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales's works, including *Shadows of...","[web serial, rational fiction, fantasy, progre...",[Damn. That was like reading some of Sanderson...,"The reviews are overwhelmingly positive, with ...",Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,Readers of r/rational would likely enjoy Alexa...,"[Brandon Sanderson, Harry Potter, Worm, Mother...",9.0,7.5,9.0,9.0,8.5,8.0,\n\n----- Thread 10262 -----\n\n## [HF][DC] Sh...,"Alexander Wales - The Metropolitan Man, Shadow...",google/gemini-flash-1.5
2,Mother of Learning,Mother of Learning is a web serial about Zoria...,"[web serial, original fiction, fantasy, time l...","[With dread but cautious optimism, Enthusiasti...",Generally very positive; praised for clever pr...,"Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Readers of r/rational will appreciate the stor...,"[Time Braid, HPMOR, Worm]",9.0,7.5,7.0,9.0,8.5,9.0,\n\n----- Thread 10262 -----\n\n## [HF] Mother...,Mother of Learning,google/gemini-flash-1.5
3,A Practical Guide to Evil,A Practical Guide to Evil is a long-running we...,"[web serial, fantasy, complete, grimdark, roma...",[It's probably one of the better written ones ...,"Generally very positive, praising the writing,...","Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,Readers of r/rational will likely enjoy this f...,"[Worm, Shadows of the Limelight]",9.5,7.0,8.0,9.0,9.0,9.0,\n\n----- Thread 10262 -----\n\n## [RT] A Prac...,A Practical Guide to Evil,google/gemini-flash-1.5
4,Worm,"Worm is a long, complete web serial (1.7 milli...","[web serial, complete, superhero, supervillain...",[I wouldn't really describe Worm as a rational...,"Worm receives overwhelmingly positive reviews,...",Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...",Readers of r/rational might enjoy Worm for its...,"[HPMOR, A Practical Guide to Evil, Other Wildb...",9.0,7.0,8.5,8.5,8.0,9.5,\n\n----- Thread 10262 -----\n\n## Worm\n\n* A...,Worm,google/gemini-flash-1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30402,r/rational Community Discussion Summary,"This is not a work of fiction, but rather a Re...","[reddit discussion, online community, rational...","[""Much less than or Not at all."", ""I don't thi...",The overall sentiment towards r/rational is mi...,Some users valued the community and its emphas...,Some users expressed negative associations wit...,Readers interested in discussions about ration...,[],3.0,7.0,0.0,0.0,0.0,0.0,\n\n----- Thread 10265 -----\n\n...mon trend r...,Much less than or Not at all.,google/gemini-flash-1.5
30403,Steelheart (The Reckoners),"Steelheart, the first book in Brandon Sanderso...","[novel, complete, superhero, superpower, decon...",[> It's another superpower deconstruction stor...,"Steelheart is mentioned in comparison to Worm,...",The book is mentioned as a related work to *Wo...,None explicitly stated; the comment thread foc...,Readers of r/rational might appreciate *Steelh...,"[Worm, Mi

In [111]:
# # also join with df4
# df_llm2 = (df_llm
#           #  .drop(columns=['url'])
#            .set_index('title2')
#            .merge(df3, left_index=True, right_index=True)
#               .sort_values('score', ascending=False)
#             #   .drop(columns=['comments'])
#             .reset_index(names='title2')
# )

# print(f'we went from {len(df3)}|{len(df_llm)} to {len(df_llm2)} rows')


# # df_llm2
# # then display as table
# # QC check correlation between score and ratings, better llm/prompts should have a higher correlation with score/karma as it's a high quality signal
# df_llm2_num = df_llm2.select_dtypes(include=['float64', 'int64'])
# df_llm2_num['score_mean'] = df_llm2_num['score'] / df_llm2_num['n_links']
# cc = df_llm2_num.iloc[:150]
# c = cc.corr()['score']
# print(f"Correlation with score ↑:\n{c} [n={len(cc)}] model={MODEL_NAME}")

### Save

In [112]:
df_llm.shape

(30407, 18)

In [119]:
df_llm = pd.DataFrame(llm_info).rename(columns={'title': 'title_llm', 'title2': 'title'})
# somehow we load dups so
print(df_llm.model.value_counts())
print(df_llm.title.value_counts())
df_llm = df_llm.drop_duplicates(subset=['title'], keep='last')
df_llm

model
google/gemini-flash-1.5    30407
Name: count, dtype: int64
title
Worth the Candle                                                                     2
Giger's 5e Table Top                                                                 2
bio page into an RSS feed                                                            2
First Flight                                                                         2
First Song Anthem                                                                    2
                                                                                    ..
one article I found that talks about a mice study                                    1
Eliezer's tumblr                                                                     1
New Acquisitions: How It Wasn’t: Game of Thrones and the Middle Ages, Part III       1
a set-up to make sure Ender can commit genocide and still be innocent of doing so    1
8.3 | Worm                                                 

,title_llm,description,tags,reviews_quotes,reviews_summary,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,rating_quality,rating_rationality,rating_writing,rating_plot,rating_character,rating_worldbuilding,context,title,model
15166,Worth the Candle,"Worth the Candle is a long, original, complete...","[web serial, LitRPG, isekai, portal fantasy, o...","[This is a self-insert litRPG portal fantasy, ...","Overall, Worth the Candle received overwhelmin...",Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,Readers of r/rational would likely enjoy Worth...,"[Mother of Learning, HPMOR, Worm, Erfworld, Lu...",9.0,7.0,8.5,8.5,8.0,9.0,\n\n----- Thread 10262 -----\n\n## [RT][WIP] W...,Worth the Candle,google/gemini-flash-1.5
15167,"Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales's works, including *Shadows of...","[web serial, rational fiction, fantasy, progre...",[Damn. That was like reading some of Sanderson...,"The reviews are overwhelmingly positive, with ...",Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,Readers of r/rational would likely enjoy Alexa...,"[Brandon Sanderson, Harry Potter, Worm, Mother...",9.0,7.5,9.0,9.0,8.5,8.0,\n\n----- Thread 10262 -----\n\n## [HF][DC] Sh...,"Alexander Wales - The Metropolitan Man, Shadow...",google/gemini-flash-1.5
15168,Mother of Learning,Mother of Learning is a web serial about Zoria...,"[web serial, original fiction, fantasy, time l...","[With dread but cautious optimism, Enthusiasti...",Generally very positive; praised for clever pr...,"Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Readers of r/rational will appreciate the stor...,"[Time Braid, HPMOR, Worm]",9.0,7.5,7.0,9.0,8.5,9.0,\n\n----- Thread 10262 -----\n\n## [HF] Mother...,Mother of Learning,google/gemini-flash-1.5
15169,A Practical Guide to Evil,A Practical Guide to Evil is a long-running we...,"[web serial, fantasy, complete, grimdark, roma...",[It's probably one of the better written ones ...,"Generally very positive, praising the writing,...","Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,Readers of r/rational will likely enjoy this f...,"[Worm, Shadows of the Limelight]",9.5,7.0,8.0,9.0,9.0,9.0,\n\n----- Thread 10262 -----\n\n## [RT] A Prac...,A Practical Guide to Evil,google/gemini-flash-1.5
15170,Worm,"Worm is a long, complete web serial (1.7 milli...","[web serial, complete, superhero, supervillain...",[I wouldn't really describe Worm as a rational...,"Worm receives overwhelmingly positive reviews,...",Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...",Readers of r/rational might enjoy Worm for its...,"[HPMOR, A Practical Guide to Evil, Other Wildb...",9.0,7.0,8.5,8.5,8.0,9.5,\n\n----- Thread 10262 -----\n\n## Worm\n\n* A...,Worm,google/gemini-flash-1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30402,r/rational Community Discussion Summary,"This is not a work of fiction, but rather a Re...","[reddit discussion, online community, rational...","[""Much less than or Not at all."", ""I don't thi...",The overall sentiment towards r/rational is mi...,Some users valued the community and its emphas...,Some users expressed negative associations wit...,Readers interested in discussions about ration...,[],3.0,7.0,0.0,0.0,0.0,0.0,\n\n----- Thread 10265 -----\n\n...mon trend r...,Much less than or Not at all.,google/gemini-flash-1.5
30403,Steelheart (The Reckoners),"Steelheart, the first book in Brandon Sanderso...","[novel, complete, superhero, superpower, decon...",[> It's another superpower deconstruction stor...,"Steelheart is mentioned in comparison to Worm,...",The book is mentioned as a related work to *Wo...,None explicitly stated; the comment thread foc...,Readers of r/rational might appreciat

In [120]:
df_llm2 = (df3
 .merge(df_llm, left_index=True, right_on='title', how='left')
 .sort_values('score', ascending=False))
print(df_llm.shape, df3.shape)
df_llm2.shape

(15241, 18) (15271, 8)


(15271, 26)

In [121]:
# print(df_llm.shape, df3.shape)
# # also join with df4
# df_llm2 = (df_llm
#           #  .drop(columns=['url'])
#            .set_index('title2')
#            .merge(df3, left_index=True, right_index=True, how='left')
#               .sort_values('score', ascending=False)
#             #   .drop(columns=['comments'])
#             .reset_index(names='title2')
# )
# df_llm2


# # now deduplicate by llm title and also keep only the ones that are fiction

In [122]:
from rrational.transform import join_uniq, chain_lists, format_flair, c2md, collapsibe, urls2a,url2a,  unique_elements 

In [123]:
# dedup by llm title
df_llm3= (
    df_llm2.groupby('title_llm').agg(
        {
            "title": "first",
            "score": "sum",
            "n_links": "sum",
            'n_comments': 'sum',
            "comments": chain_lists,
            "thread_urls": chain_lists,
            'first_link_utc': 'min',
            'last_link_utc': 'max',
            'url': chain_lists,
            # 'title2': list,
              'description': 'first',
                'tags': chain_lists,
                'reviews_quotes': 'first',
       'reviews_summary': 'first',
       'reccomendations': 'first',
       'disrecommendations': 'first',
         'why': 'first',
       'if_you_liked_x_you_will_like_this': 'first',
         'rating_quality': 'first',
       'rating_rationality': 'first',
         'rating_writing': 'first',
         'rating_plot': 'first',
       'rating_character': 'first',
         'rating_worldbuilding': 'first',
        #    'context', 
        #    'model'
        }
    )
    .sort_values("score", ascending=False)
    .reset_index()
)
df_llm3

,title_llm,title,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url,...,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,rating_quality,rating_rationality,rating_writing,rating_plot,rating_character,rating_worldbuilding
0,Worth the Candle,Worth the Candle,6082,329,329,"[{'author': 'Noumero', 'author_flair_text': 'S...",[https://reddit.com/r/rational/comments/7dz6kj...,2015-06-05 21:38:35,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...,...,Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,Readers of r/rational would likely enjoy Worth...,"[Mother of Learning, HPMOR, Worm, Erfworld, Lu...",9.0,7.0,8.5,8.5,8.0,9.0
1,Mother of Learning,Mother of Learning,2055,187,187,"[{'author': 'TimeLoopedPowerGamer', 'author_fl...",[https://reddit.com/r/rational/comments/cmc4a0...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...,...,"Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Readers of r/rational will appreciate the stor...,"[Time Braid, HPMOR, Worm]",9.0,7.5,7.0,9.0,8.5,9.0
2,"Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales - The Metropolitan Man, Shadow...",1402,22,22,"[{'author': 'alexanderwales', 'author_flair_te...",[https://reddit.com/r/rational/comments/al7z2v...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales],...,Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,Readers of r/rational would likely enjoy Alexa...,"[Brandon Sanderson, Harry Potter, Worm, Mother...",9.0,7.5,9.0,9.0,8.5,8.0
3,A Practical Guide to Evil,A Practical Guide to Evil,1276,139,139,"[{'author': 'Escapement', 'author_flair_text':...",[https://reddit.com/r/rational/comments/byyy3d...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ...",...,"Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,Readers of r/rational will likely enjoy this f...,"[Worm, Shadows of the Limelight]",9.5,7.0,8.0,9.0,9.0,9.0
4,Worm,Worm,1014,97,97,"[{'author': 'ArgentStonecutter', 'author_flair...",[https://reddit.com/r/rational/comments/16rsx5...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa...",...,Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...",Readers of r/rational might enjoy Worm for its...,"[HPMOR, A Practical Guide to Evil, Other Wildb...",9.0,7.0,8.5,8.5,8.0,9.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13407,r/rational Community Discussion Summary,Much less than or Not at all.,-7,1,1,"[{'author': 'Blusqere', 'author_flair_text': N...",[https://reddit.com/r/rational/comments/kaat79...,2020-12-10 16:35:49,2020-12-10 16:35:49,[https://www.merriam-webster.com/dictionary/no...,...,Some users valued the community and its emphas...,Some users expressed negative associations wit...,Readers interested in discussions about ration...,[],3.0,7.0,0.0,0.0,0.0,0.0
13408,Steelheart (The Reckoners),Steelheart (The Reckoners): 9780385743570: San...,-10,1,1,"[{'author': 'ben_oni', 'author_flair_text': No...",[https://reddit.com/r/rational/comments/7cneg5...,2017-11-13 20:58:01,2017-11-13 20:58:01,[https://www.amazon.com/Steelheart-Reckoners-B...,...,The book is mentioned as a related work to *Wo...,None explicitly stated; the comment thread foc...,Readers of r/rational might appreciate *Steelh...,"[Worm, Mistborn, The Reckoners, The Incredible...",7.0,7.0,7.0,7.0,6.0,8.0
13409,Orthogonality Thesis,Orthogonality thesis,-10,2,2,"[{'author': 'BadGoyWithAGun', 'author_flair_te...",[https://reddit.com/r/rational/comments/3vc0si...,2015-12-04 18:17:07,2016-07-11 16:27:30,[https://wiki.lesswrong.com/wiki/Orthogonality...,...,Users who appreciate discussions of Frie

In [124]:
df_llm3.to_parquet("../outputs/df_links5.parquet")